# Compare base-pay model specifications

Use the same production snapshot and periods to compare employee-aligned,
grouped rate/staffing, and effective-rate models. All three must reproduce
the same actual payroll. Different attribution definitions need not agree.

Install the package with `[notebook]` extras. Keep `comparison_models.py`
beside this notebook; edit its common payroll declarations for your data.
The example follows GUIDE.MD: permanent rate × hours plus bonus, all
pre-regional, and temporary regular/overtime pay. It is not an automatic
converter for arbitrary models. No input data or outputs are stored here.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import labor_cost
from labor_cost import (
    AnalysisRequest,
    Catalog,
    export_run,
    render_model,
    report,
)

catalog_path = None
dataset_id = "payroll"
dataset_version = None
period_a = None
period_b = None
source_cost_column = None  # Prefer an independent actual payroll total.
exposure_column = None  # None uses the model's default exposure policy.
explainer = "exact"
n_permutations = 256
random_seed = 0
selected_teams = []
currency_label = "Currency"
report_renderer = "plotly_mimetype"
export_directory = None  # Optional new run bundles, one per variant.
example_directory = None  # Folder containing comparison_models.py.
source_schema = None  # Optional SourceSchema with your key/axis mappings.
field_columns = {}  # Model field ID -> your source column.
permanent_category = "permanent"
temporary_category = "temporary"

# Presentation settings: rerun reporting without attribution.
report_chart = "impacts"  # Or "waterfall".
report_top_n = None  # Set 5 for the largest impacts plus Other.


In [ ]:
locations = ([Path(example_directory)] if example_directory else [
    Path.cwd(), Path.cwd() / "production" / "examples",
    Path(labor_cost.__file__).resolve().parents[2] / "examples",
])
example_path = next(
    (p / "comparison_models.py" for p in locations
     if (p / "comparison_models.py").is_file()), None,
)
if example_path is None:
    raise FileNotFoundError(
        "Set example_directory to the folder containing comparison_models.py"
    )
spec = importlib.util.spec_from_file_location("pay_models", example_path)
declarations = importlib.util.module_from_spec(spec)
spec.loader.exec_module(declarations)
models = declarations.comparison_models(
    source=source_schema, columns=field_columns,
    permanent_category=permanent_category,
    temporary_category=temporary_category,
)
for name, model in models.items():
    print(name, model.model_id, model.version, model.fingerprint)
    display(render_model(model))

## Interpret the alternatives

- **Aligned:** rate and time changes follow matched employees; membership
  separately captures which profiles contribute and their exposure conversions.
- **Grouped:** rate and membership switch together; employee time remains
  separate. This is a new game, not a sum of the old bars.
- **Effective:** rate is weighted by worked hours; quantity is hours per
  exposure. Reassignment of hours can change effective rate without raises.
  A populated zero-hour cell uses its exposure-weighted contractual rate,
  recorded as an assumption. Its observed hours and base payroll remain zero.

Composition, bonuses, and temporary payroll use common declarations.
For monthly salaries use worked fractions, corresponding units, and accurate
labels instead of calling fractions hours. Exposure is not automatically FTE.

In [ ]:
results = {}
summary = pd.DataFrame()
if all(x is not None for x in (
    catalog_path, dataset_version, period_a, period_b,
)):
    dataset = Catalog(Path(catalog_path)).dataset(dataset_id, dataset_version)
    request = AnalysisRequest(
        period_a, period_b, source_cost_column=source_cost_column,
        exposure_column=exposure_column, explainer=explainer,
        n_permutations=n_permutations, random_seed=random_seed,
    )
    # Sequential runs preserve estimator RNG isolation.
    for name, model in models.items():
        print("Analyzing", name)
        results[name] = labor_cost.analyze(model, dataset, request)
    records = []
    baseline = results["aligned"]
    for name, result in results.items():
        assert result.manifest["dataset"] == baseline.manifest["dataset"]
        np.testing.assert_allclose(
            result.table("canonical_data").row_cost,
            baseline.table("canonical_data").row_cost,
            rtol=1e-10, atol=1e-6,
        )
        teams = result.table("team_summary")
        records.append({
            "variant": name,
            "model_fingerprint": result.manifest["model_fingerprint"],
            "source_fingerprint": result.manifest["dataset"][
                "selected_fingerprint"
            ],
            "cost_a": teams.period_A_cost.sum(),
            "cost_b": teams.period_B_cost.sum(),
            "seconds": result.manifest["elapsed_seconds"],
            "validated": result.table("validation").passed.all(),
        })
    summary = pd.DataFrame(records).set_index("variant")
    np.testing.assert_allclose(
        summary[["cost_a", "cost_b"]].to_numpy(),
        np.tile(summary.loc["aligned", ["cost_a", "cost_b"]]
                .to_numpy(dtype=float), (len(summary), 1)),
        rtol=1e-10, atol=1e-6,
    )
    display(summary)
else:
    print("Set the catalog, dataset version, and periods to compare payroll.")

## Inspect calculated results

Rerun only the following cell to change team selections or chart styling.
Driver tables remain separate: an absent driver is not a zero contribution.
Organizational allocations and local-team allocations are different views.

In [ ]:
reports = {}
for name, result in results.items():
    print(name)
    display(result.table("driver_bridge"))
    display(result.table("coalition_audit"))
    display(result.table("assumptions"))
    reports[name] = report(
        result, teams=selected_teams, currency_label=currency_label,
        show=False, chart=report_chart, top_n=report_top_n,
    )
    reports[name].show(renderer=report_renderer)
if "effective" in results:
    display(results["effective"].table("diagnostics/aggregate_factors"))

In [ ]:
if export_directory is not None:
    for name, result in results.items():
        destination = Path(export_directory) / (name + "-" + result.run_id)
        export_run(result, destination)
        print("Saved", destination)